In [21]:
using PowerSystems
using PowerSimulations
using PowerGraphics
using PowerAnalytics
using HiGHS
using CSV
using DataFrames
using TimeSeries
using Dates
using Statistics
using StorageSystemsSimulations
using Plots

data_directory = "./1_three_zones";

# read in relevant CSV files

storage_df = CSV.read(joinpath(data_directory, "resources", "Storage.csv"), DataFrame) ;
thermal_df = CSV.read(joinpath(data_directory, "resources", "Thermal.csv"), DataFrame) ;
thermal_df_plc = CSV.read(joinpath(data_directory, "resources", "ThermalPLC.csv"), DataFrame) ;
vre_df = CSV.read(joinpath(data_directory, "resources", "Vre.csv"), DataFrame) ;
#hydro_df = CSV.read(joinpath(data_directory, "resources", "Hydro.csv"), DataFrame) ;
#hydrores_df = CSV.read(joinpath(data_directory, "resources", "HydroReservoir.csv"), DataFrame) ;
#hydropumped_df = CSV.read(joinpath(data_directory, "resources", "HydroPump.csv"), DataFrame) ;
capacity_df = CSV.read(joinpath(data_directory, "results", "capacity.csv"), DataFrame);
demand_df = CSV.read(joinpath(data_directory, "system", "Demand_data.csv"), DataFrame) ;
network_df = CSV.read(joinpath(data_directory, "system", "Network.csv"), DataFrame) ;
network_expansion_df = CSV.read(joinpath(data_directory, "results", "network_expansion.csv"), DataFrame) ;
fuels_df = CSV.read(joinpath(data_directory, "system", "Fuels_data.csv"), DataFrame) ;
gen_variability_df = CSV.read(joinpath(data_directory, "system", "Generators_variability.csv"), DataFrame) ;
mt_df = CSV.read(joinpath(data_directory, "MoverTypesMapping.csv"), DataFrame) ;
fm_df = CSV.read(joinpath(data_directory, "FuelMapping.csv"), DataFrame) ;
sm_df = CSV.read(joinpath(data_directory, "StorageMapping.csv"), DataFrame) ;
rm_df = CSV.read(joinpath(data_directory, "RenewableMapping.csv"), DataFrame) ;

# create 3 dictionaries, one for fuel types, one for mover types, one for storage types
mover_dict = Dict((row.Key) => row.Value for row in eachrow(mt_df)) ;
fuel_dict = Dict((row.Key) => row.Value for row in eachrow(fm_df)) ;
storage_dict = Dict((row.Key) => row.Value for row in eachrow(sm_df)) ;
renewable_dict = Dict((row.Key) => row.Value for row in eachrow(rm_df)) ;

#fuel costs:
column_names = names(fuels_df)
columns_to_read = column_names[2:end]
fuel_prices = Dict{String, Union{Float64, Missing}}()
for col in columns_to_read
    average = mean(fuels_df[2:end, Symbol(col)])  # Ignore the first row
    fuel_prices[String(col)] = average
end

In [19]:
# activate julia environment in a controlled way
julia_environment = "Julia_src/activate.jl"
include(julia_environment)

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.to

Activating the Julia virtual environment


In [20]:
Pkg.status()

Status `~/.julia/environments/v1.10/Project.toml`
  [6e4b80f9] BenchmarkTools v1.6.0
  [336ed68f] CSV v0.10.15
  [a93c6f00] DataFrames v1.7.0
  [864edb3b] DataStructures v0.18.20
  [5789e2e9] FileIO v1.16.6
  [60bf3e95] GLPK v1.2.1
  [5d317b1e] GenX v0.4.2
  [5c1252a2] GeometryBasics v0.5.1
  [2e9cd046] Gurobi v1.6.0
  [87dc4568] HiGHS v1.13.0
  [7073ff75] IJulia v1.26.0
  [916415d5] Images v0.26.1
  [b6b21f68] Ipopt v1.7.1
  [4076af6c] JuMP v1.23.6
  [add582a8] MLJ v0.20.7
  [b8f27783] MathOptInterface v1.35.0
  [fdba3010] MathProgBase v0.7.8
  [91a5bcdd] Plots v1.40.9
  [56ce1300] PowerAnalytics v0.8.1
  [5f7eddb3] PowerGraphics v0.19.0
  [e690365d] PowerSimulations v0.29.1
  [bcd98974] PowerSystems v4.4.1
  [731186ca] RecursiveArrayTools v3.27.4
  [295af30f] Revise v3.7.1
  [82193955] SCIP v0.12.1
  [2913bbd2] StatsBase v0.34.4
  [e2f1a126] StorageSystemsSimulations v0.11.1
  [9e3dc215] TimeSeries v0.24.2
  [fdbf4ff8] XLSX v0.10.4
  [ddb6d928] YAML v0.4.12
  [ade2ca70] Dates
  [37e2

In [23]:
start_time=1
end_time=24

# adding nodes
nodes=[]
for i in 1:count(!ismissing, network_df[:, "Network_zones"])
    bus=ACBus(;
        number=i,
        name= network_df[i, :1],
        bustype= i == 1 ? "REF" : "PQ",
        angle=0,
        magnitude=0,
        voltage_limits=nothing,
        base_voltage=230
    )
    push!(nodes,bus)
end
nodes=Vector{ACBus}(nodes)

# make system
sys_base_power=100.0
sys=System(sys_base_power,nodes)
set_units_base_system!(sys, "NATURAL_UNITS") ;

areas=[]
for i in 1:count(!ismissing, network_df[:, "Network_zones"])
    area_factor=Area(;name=network_df[i, :1])
    push!(areas,area_factor)
    set_area!(nodes[i], areas[i])
end
add_components!(sys,areas)

# create line vector
lines=[]
for i in 1:count(!ismissing, network_df[:, "Network_Lines"])
    existing_cap = network_df[i, :Line_Max_Flow_MW]
    new_cap = network_expansion_df[network_expansion_df.Line .== network_df[i, :Network_Lines], "New_Trans_Capacity"][1]
    line = Line(;
        name = string(network_df[i, :Network_Lines]),
        available = true,
        active_power_flow = 0.0,
        reactive_power_flow = 0.0,
        arc=Arc(from = nodes[network_df[i, :Start_Zone]], to = nodes[network_df[i, :End_Zone]]),
        r=0.0,
        x=0.0,
        b= (from = 0.0, to = 0.0),
        rating = (existing_cap+new_cap)/sys_base_power,
        angle_limits = (0.0, 0.0),
    )
    push!(lines,line)  
end
add_components!(sys,lines) # add lines vector to system

ais=[]
for i in 1:count(!ismissing, network_df[:,"Network_Lines"])
    ai=AreaInterchange(;
        name = string(network_df[i, :Network_Lines]),
        available = true,
        active_power_flow = 0.0,
        flow_limits=(from_to=get_rating(lines[i]),to_from=get_rating(lines[i])),
        from_area=areas[network_df[i, :Start_Zone]],
        to_area=areas[network_df[i, :End_Zone]],
    )
    push!(ais,ai)
end
add_components!(sys,ais)

#=# thermal generators
thermal_gen=[]
for i in 1:count(!ismissing, thermal_df[:, "Resource"])
    resource_name = thermal_df[i, :Resource]
    capacity_mw = capacity_df[capacity_df.Resource .== resource_name, :][1, :EndCap]
    thermal = ThermalStandard(;
        name = resource_name,
        available = true,
        status = true,
        bus = nodes[thermal_df[i, :Zone]],
        active_power = 0,
        reactive_power = 0,
        rating = 1,
        active_power_limits = (min = thermal_df[i, :Min_Power], max = 1),
        reactive_power_limits = nothing,
        ramp_limits = (up = 1e9,down=1e9),#thermal_df[i, :Ramp_Up_Percentage], down = thermal_df[i, :Ramp_Dn_Percentage]),
        operation_cost = ThermalGenerationCost(
            variable = FuelCurve(
                value_curve = LinearCurve(thermal_df[i, :Heat_Rate_MMBTU_per_MWh]),
                fuel_cost = fuel_prices[string(thermal_df[i, :Fuel])]
                ),
            fixed = 0,
            start_up = thermal_df[i, :Start_Cost_per_MW]*capacity_mw,
            shut_down = 0.0,
            ),
        base_power = capacity_mw,
        time_limits = (up = thermal_df[i, :Up_Time], down = thermal_df[i, :Down_Time]),
        must_run = false,
        prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[thermal_df[i, :Resource]])),
        fuel = fuel_dict[string(thermal_df[i, :Fuel])],
    )
    push!(thermal_gen, thermal)
end
add_components!(sys,thermal_gen)=#


# thermal generators with piecewise linear cost curve
thermal_gen_plc=[]
for i in 1:count(!ismissing, thermal_df_plc[:, "Resource"])
    resource_name = thermal_df_plc[i, :Resource]
    capacity_mw = capacity_df[capacity_df.Resource .== resource_name, :][1, :EndCap]
    thermal_plc = ThermalStandard(;
        name = resource_name,
        available = true,
        status = true,
        bus = nodes[thermal_df_plc[i, :Zone]],
        active_power = 0,
        reactive_power = 0,
        rating = 1,
        active_power_limits = (min = thermal_df_plc[i, :Min_Power], max = 1),
        reactive_power_limits = nothing,
        ramp_limits = (up = 1e9,down=1e9),#thermal_df[i, :Ramp_Up_Percentage], down = thermal_df[i, :Ramp_Dn_Percentage]),
        operation_cost = ThermalGenerationCost(
            variable = FuelCurve(
                value_curve = PiecewisePointCurve([(thermal_df_plc[i, :Min_Power]*capacity_mw, thermal_df_plc[i, :Heat_Rate_MMBTU_per_MWh_1]), 
                (thermal_df_plc[i, :Intermediate_Power]*capacity_mw, thermal_df_plc[i, :Heat_Rate_MMBTU_per_MWh_2]), 
                (capacity_mw, thermal_df_plc[i, :Heat_Rate_MMBTU_per_MWh_3])]),
                fuel_cost = fuel_prices[string(thermal_df_plc[i, :Fuel])]
                ),
            fixed = 0,
            start_up = thermal_df_plc[i, :Start_Cost_per_MW]*capacity_mw,
            shut_down = 0.0,
            ),
        base_power = capacity_mw,
        time_limits = (up = thermal_df_plc[i, :Up_Time], down = thermal_df_plc[i, :Down_Time]),
        must_run = false,
        prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[thermal_df_plc[i, :Resource]])),
        fuel = fuel_dict[string(thermal_df_plc[i, :Fuel])],
    )
    push!(thermal_gen_plc, thermal_plc)
end
add_components!(sys,thermal_gen_plc)

# renewable generators
for i in 1:count(!ismissing, vre_df[:, "Resource"])
    resource_name = vre_df[i, :Resource]
    renewable = RenewableDispatch(;
        name = resource_name,
        available = true,
        bus = nodes[vre_df[i, :Zone]],
        active_power = 0.0,
        reactive_power = 0.0,
        rating = 1,
        prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[resource_name])),
        reactive_power_limits = (min = 0.0, max = 0.0),
        power_factor = 1.0,
        operation_cost = RenewableGenerationCost(CostCurve(LinearCurve(vre_df[i, :Var_OM_Cost_per_MWh]))),
        base_power = capacity_df[capacity_df.Resource .== resource_name, :][1, :EndCap]
        )
    add_component!(sys,renewable)
    ren_values = gen_variability_df[start_time:end_time, Symbol(resource_name)]
    timestamps = range(DateTime("2020-01-01T00:00:00"), step = Dates.Hour(1), length = 24);
    ren_timearray = TimeArray(timestamps, ren_values);
    ren_time_series = SingleTimeSeries(
        name = "max_active_power",
        data = ren_timearray;
        scaling_factor_multiplier = get_max_active_power
    );
    add_time_series!(sys, renewable, ren_time_series);
end

#=
for i in 1:count(!ismissing, hydro_df[:, "Resource"])
    resource_name = hydro_df[i, :Resource]
    hydro = HydroDispatch(;
            name = resource_name,
            available = true,
            bus = nodes[hydro_df[i, :Zone]],
            active_power = 0.0, # Per-unitized by device base_power
            reactive_power = 0.0, # Per-unitized by device base_power
            rating = 1.0, # 5 MW per-unitized by device base_power
            prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[resource_name])),
            active_power_limits = (min = hydro_df[i, :Min_Power], max = 1.0), # 0 MW to 5 MW per-unitized by device base_power
            reactive_power_limits = (min = 0.0, max = 0.0), # 0 MVAR to 0.25 MVAR per-unitized by device base_power
            ramp_limits = (up = 1e9,down=1e9),
            time_limits = nothing,
            base_power = 100.0,
            operation_cost = HydroGenerationCost(
                variable = FuelCurve(
                value_curve = LinearCurve(hydro_df[i, :Var_OM_Cost_per_MWh]),
                fuel_cost = 0.0
                ), fixed = hydro_df[i, :Fixed_OM_Cost_per_MWhyr])
    )
    add_component!(sys,hydro)
    hydro_values = gen_variability_df[start_time:end_time, Symbol(resource_name)]
    timestamps = range(DateTime("2020-01-01T00:00:00"), step = Dates.Hour(1), length = 24);
    hydro_timearray = TimeArray(timestamps, hydro_values);
    hydro_time_series = SingleTimeSeries(
        name = "max_active_power",
        data = hydro_timearray;
        scaling_factor_multiplier = get_max_active_power
    );
    add_time_series!(sys, hydro, hydro_time_series);
end

for i in 1:count(!ismissing, hydrores_df[:, "Resource"])
        resource_name = hydrores_df[i, :Resource]
        hydrores = HydroEnergyReservoir(; 
            name = resource_name,
            available = true,
            bus = nodes[hydrores_df[i, :Zone]], 
            active_power = 0.0,
            reactive_power = 0.0,
            rating = 1.0,
            prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[resource_name])),
            active_power_limits = MinMax(min = 0.0, max = 1.0),
            reactive_power_limits = nothing,
            ramp_limits = nothing,
            time_limits = nothing,
            base_power = 1.0,
            storage_capacity = MinMax(min = 0.0, max = 1.0),
            inflow = 0.0,
            initial_storage = MinMax(min = 0.0, max = 1.0),
            operation_cost = HydroGenerationCost(
                variable = CostCurve(LinearCurve(hydrores_df[i, :Var_OM_Cost_per_MWh])),
                fixed = 0.0,
                start_up = 0.0,
                shut_down = 0.0,
                energy_shortage_cost = 0.0,
                energy_surplus_cost = 0.0
            ),
            storage_target = MinMax(min = 0.0, max = 1.0),
            conversion_factor = 1.0,
            status = PumpHydroStatus.OFF,
            time_at_status = 0.0,
            services = [],
            dynamic_injector = nothing,
            ext = Dict{String, Any}(),
            internal = InfrastructureSystemsInternal()
    );
        
end


for i in 1:count(!ismissing, hydropumped_df[:, "Resource"])
        resource_name = hydropumped_df[i, :Resource]
        hydropumped = HydroPumpedStorage(;
            name = resource_name,
            available = true,
            bus = nodes[hydropumped_df[i, :Zone]],
            active_power = 0.0,
            reactive_power = 0.0,
            rating = 1.0,
            base_power = 1.0,
            prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[resource_name])),
            active_power_limits = MinMax(min = 0.0, max = 1.0),
            reactive_power_limits = nothing,
            ramp_limits = nothing,
            time_limits = nothing,
            rating_pump = 1.0,
            active_power_limits_pump = MinMax(min = 0.0, max = 1.0),
            reactive_power_limits_pump = nothing,
            ramp_limits_pump = nothing,
            time_limits_pump = nothing,
            storage_capacity = MinMax(min = 0.0, max = 1.0),
            inflow = 0.0,
            outflow = 0.0,
            initial_storage = MinMax(min = 0.0, max = 1.0),
            storage_target = MinMax(min = 0.0, max = 1.0),
            operation_cost = HydroGenerationCost(
                variable = CostCurve(LinearCurve(hydropumped_df[i, :Var_OM_Cost_per_MWh])),
                fixed = 0.0,
                start_up = 0.0,
                shut_down = 0.0,
                energy_shortage_cost = 0.0,
                energy_surplus_cost = 0.0
            ),
            pump_efficiency = 1.0,
            conversion_factor = 1.0,
            status = PumpHydroStatus.OFF,
            time_at_status = 0.0,
            services = [],
            dynamic_injector = nothing,
            ext = Dict{String, Any}(),
            internal = InfrastructureSystemsInternal()
        );
end=#


# storage
for i in 1:count(!ismissing, storage_df[:, "Resource"])
    resource_name = storage_df[i, :Resource]
    storage_device=EnergyReservoirStorage(;
        name = resource_name,
        available = true,
        bus = nodes[storage_df[i, :Zone]],
        prime_mover_type = getproperty(PrimeMovers,Symbol(mover_dict[resource_name])),
        storage_technology_type = getproperty(StorageTech,Symbol(storage_dict[resource_name])),
        storage_capacity = capacity_df[capacity_df.Resource .== resource_name, :][1, :EndEnergyCap],
        storage_level_limits = (min = 0.0, max = 1.0),
        initial_storage_capacity_level = 0.0,
        rating = 1.0,
        active_power = 0.0,
        input_active_power_limits = (min = 0.0, max = 1.0),
        output_active_power_limits = (min = 0.0, max = 1.0),
        efficiency = (in = storage_df[i, :Eff_Up], out = storage_df[i, :Eff_Down]),
        reactive_power = 0,
        reactive_power_limits = (min = 0.0, max = 0.0),
        base_power = capacity_df[capacity_df.Resource .== resource_name, :][1, :EndCap],
        operation_cost = StorageCost(charge_variable_cost = CostCurve(LinearCurve(storage_df[i, :Var_OM_Cost_per_MWh_In])), 
            discharge_variable_cost = CostCurve(LinearCurve(storage_df[i, :Var_OM_Cost_per_MWh])), 
            fixed = 0.0, start_up = 0.0, shut_down = 0.0, energy_shortage_cost = 0.0, 
            energy_surplus_cost = 0.0),
        )
    add_component!(sys, storage_device)
end
#=
for i in 1:count(!ismissing, resup_df[:, "Resource"])
    resource_name = resup_df[i, :Resource]
    res_up = ConstantReserve{ReserveUp}(;
        name = resource_name,
        available = true,
        time_frame = 0.0,
        requirement = 0.0,
        sustained_time = 0.0,
        max_output_fraction = 0.0,
        max_participation_factor = 0.0,
        deployed_fraction = 0.0,
        ext = Dict{String, Any}(),
        internal = InfrastructureSystemsInternal()
    );
    
end

for i in 1:count(!ismissing, resdown_df[:, "Resource"])
    resource_name = resdown_df[i, :Resource]
    res_down = ConstantReserve{ReserveDown}(;
        name = resource_name,
        available = true,
        time_frame = 0.0,
        requirement = 0.0,
        sustained_time = 0.0,
        max_output_fraction = 0.0,
        max_participation_factor = 0.0,
        deployed_fraction = 0.0,
        ext = Dict{String, Any}(),
        internal = InfrastructureSystemsInternal()
    );
    
end

for i in 1:count(!ismissing, ressym_df[:, "Resource"])
    resource_name = ressym_df[i, :Resource]
    ressym_up = ConstantReserve{ReserveSymmetric}(;
        name = resource_name,
        available = true,
        time_frame = 0.0,
        requirement = 0.0,
        sustained_time = 0.0,
        max_output_fraction = 0.0,
        max_participation_factor = 0.0,
        deployed_fraction = 0.0,
        ext = Dict{String, Any}(),
        internal = InfrastructureSystemsInternal()
    );
    
end
=#

# adding load to each zone:
for i in 1:count(!ismissing, network_df[:, "Network_zones"])
    zone_name = "Load_z"*string(i)
    power_load=InterruptiblePowerLoad(;
        name=zone_name,
        available=true,
        bus=nodes[i],
        active_power=1e9,
        reactive_power=1e9,
        base_power=1,
        max_active_power=1,
        max_reactive_power=0,
        operation_cost=LoadCost(variable=CostCurve(LinearCurve(9000)),fixed=0)
        )
    add_component!(sys, power_load)
    dates = range(DateTime("2020-01-01T00:00:00"), step = Dates.Hour(1), length = 24)
    data = TimeArray(dates, Float64.(demand_df[start_time:end_time,Symbol("Demand_MW_z" * "" * string(i))]))
    time_series = SingleTimeSeries(name="max_active_power", data=data,scaling_factor_multiplier=get_max_active_power)
    add_time_series!(sys,get_component(InterruptiblePowerLoad, sys, zone_name),time_series)
end

transform_single_time_series!(sys,Dates.Hour(24),Dates.Hour(24)) 

template_uc=ProblemTemplate()
set_device_model!(template_uc,ThermalStandard,ThermalDispatchNoMin)
set_device_model!(template_uc,InterruptiblePowerLoad,PowerLoadDispatch)
#set_device_model!(template_uc, HydroDispatch, HydroDispatchRunOfRiver)
set_device_model!(template_uc, RenewableDispatch, RenewableFullDispatch)
set_device_model!(template_uc, Line, StaticBranch)
set_device_model!(template_uc,DeviceModel(
        EnergyReservoirStorage,
        StorageDispatchWithReserves;
        attributes=Dict("reservation" => true,
            "cycling_limits" => false,
            "energy_target" => false,
            "complete_coverage" => false,
            "regularization" => true,
        ),
        use_slacks=false,
    )
)
set_network_model!(template_uc, NetworkModel(AreaBalancePowerModel))
set_device_model!(template_uc, AreaInterchange, StaticBranch)
problem=DecisionModel(template_uc,sys;optimizer=optimizer_with_attributes(HiGHS.Optimizer),horizon=Dates.Hour(24))

build!(problem,output_dir=mktempdir())
solve!(problem)
res=OptimizationProblemResults(problem) ;
gen = get_generation_data(res) ;

# get model data
charge_energyreservoirstorage=sum(Matrix(gen.data[:ActivePowerInVariable__EnergyReservoirStorage][:,2:end]);dims=2)[:]
discharge_energyreservoirstorage=sum(Matrix(gen.data[:ActivePowerOutVariable__EnergyReservoirStorage][:,2:end]);dims=2)[:]
activepower_thermalstandard=sum(Matrix(gen.data[:ActivePowerVariable__ThermalStandard][:,2:end]);dims=2)[:]
df_ren_dis=gen.data[:ActivePowerVariable__RenewableDispatch]
activepower_wind=sum(df_ren_dis[!, colname] for colname in names(df_ren_dis)[2:end] if renewable_dict[colname] == "wind")
activepower_solar=sum(df_ren_dis[!, colname] for colname in names(df_ren_dis)[2:end] if renewable_dict[colname] == "solar")
demand=sum(Matrix(demand_df[start_time:end_time,(end-(length(areas)-1)):end]);dims=2)[:]
curtailment=sum(Matrix(gen.data[:ActivePowerVariable__RenewableDispatch__Curtailment][:,2:end]);dims=2)[:]

#prepare for plotting
thermal_and_wind = activepower_thermalstandard + activepower_wind
thermal_wind_solar = thermal_and_wind + activepower_solar
thermal_wind_solar_storage = thermal_wind_solar + discharge_energyreservoirstorage
curtailment_total = thermal_wind_solar_storage + curtailment

x_data = start_time:end_time 
plot(x_data, charge_energyreservoirstorage*-1, seriestype=:path, fillrange=0, fillalpha=0.3, label="Storage In", xlabel="Time (hours)", ylabel="MegaWatts", color=:purple)
plot!(x_data, demand, seriestype=:path, label="Load", color=:black)
plot!(x_data, activepower_thermalstandard, seriestype=:path, fillrange=0, fillalpha=0.3, label="Thermal", color=:black)
plot!(x_data, thermal_and_wind, seriestype=:path, fillrange=(activepower_thermalstandard), fillalpha=0.2, label="Wind Data", color=:blue)
plot!(x_data, thermal_wind_solar, seriestype=:path, fillrange=(thermal_and_wind), fillalpha=0.3, label="Solar Data", color=:yellow)
plot!(x_data, thermal_wind_solar_storage, seriestype=:path, fillrange=(thermal_wind_solar), fillalpha=0.3, label="Storage out", color=:purple)
plot!(x_data, curtailment_total, seriestype=:path, fillrange=(thermal_wind_solar_storage), fillalpha=0.3, label="Curtailment", color=:red)


┌ Warning: There are no Generator Components in the System
└ @ PowerSystems /Users/sc87/.julia/packages/PowerSystems/vi3a3/src/utils/IO/system_checks.jl:59
┌ Warning: There are no ElectricLoad Components in the System
└ @ PowerSystems /Users/sc87/.julia/packages/PowerSystems/vi3a3/src/utils/IO/system_checks.jl:59
┌ Info: Unit System changed to UnitSystem.NATURAL_UNITS = 2
└ @ PowerSystems /Users/sc87/.julia/packages/PowerSystems/vi3a3/src/base.jl:491
┌ Warning: There is only one forecast window. Setting interval = empty period
└ @ InfrastructureSystems /Users/sc87/.julia/packages/InfrastructureSystems/rXaFr/src/system_data.jl:654
┌ Warning: There is only one forecast window. Setting interval = empty period
└ @ InfrastructureSystems /Users/sc87/.julia/packages/InfrastructureSystems/rXaFr/src/system_data.jl:654
┌ Warning: There is only one forecast window. Setting interval = empty period
└ @ InfrastructureSystems /Users/sc87/.julia/packages/InfrastructureSystems/rXaFr/src/system_data.jl:

ErrorException: build! of the DecisionModel{GenericOpProblem} GenericOpProblem failed: InfrastructureSystems.Optimization.ModelBuildStatusModule.ModelBuildStatus.FAILED = 1

Status `~/.julia/environments/v1.10/Project.toml`
⌃ [6e4b80f9] BenchmarkTools v1.5.0
  [336ed68f] CSV v0.10.15
  [a93c6f00] DataFrames v1.7.0
  [864edb3b] DataStructures v0.18.20
  [5789e2e9] FileIO v1.16.6
  [60bf3e95] GLPK v1.2.1
⌃ [5d317b1e] GenX v0.4.1
  [5c1252a2] GeometryBasics v0.5.1
  [2e9cd046] Gurobi v1.6.0
⌃ [87dc4568] HiGHS v1.12.2
  [7073ff75] IJulia v1.26.0
  [916415d5] Images v0.26.1
  [b6b21f68] Ipopt v1.7.1
⌃ [4076af6c] JuMP v1.23.5
  [add582a8] MLJ v0.20.7
⌃ [b8f27783] MathOptInterface v1.34.0
  [fdba3010] MathProgBase v0.7.8
  [91a5bcdd] Plots v1.40.9
⌅ [56ce1300] PowerAnalytics v0.7.0
⌃ [5f7eddb3] PowerGraphics v0.18.0
⌅ [e690365d] PowerSimulations v0.28.3
⌃ [bcd98974] PowerSystems v4.1.4
  [731186ca] RecursiveArrayTools v3.27.4
⌃ [295af30f] Revise v3.6.4
⌃ [82193955] SCIP v0.11.6
  [2913bbd2] StatsBase v0.34.4
⌃ [e2f1a126] StorageSystemsSimulations v0.10.2
  [9e3dc215] TimeSeries v0.24.2
  [fdbf4ff8] XLSX v0.10.4
  [ddb6d928] YAML v0.4.12
  [ade2ca70] Dates
  [37e2

In [9]:
# compare to GenX

# Read in relevant data from genx outputs:
data_directory = "./1_three_zones_prodsim";
demand_df = CSV.read(joinpath(data_directory, "system", "Demand_data.csv"), DataFrame) ;
storage_df = CSV.read(joinpath(data_directory, "resources", "Storage.csv"), DataFrame) ;
charge_df = CSV.read(joinpath(data_directory, "results", "charge.csv"), DataFrame)[3:end,:] ;
power = CSV.read(joinpath(data_directory, "results", "power.csv"), DataFrame)[3:end,:] ;
curtailment_df = CSV.read(joinpath(data_directory, "results", "curtail.csv"), DataFrame)[3:end,:] ;
thermal_df = CSV.read(joinpath(data_directory, "resources", "Thermal.csv"), DataFrame) ;
renewable_df = CSV.read(joinpath(data_directory, "resources", "Vre.csv"), DataFrame) ;
renewablemapping_df = CSV.read(joinpath(data_directory, "RenMap.csv"), DataFrame) ;
renewables_dict = Dict((row.Key) => row.Value for row in eachrow(renewablemapping_df)) ;

demand=sum(Matrix(demand_df[start_time:end_time,(end-(length(areas)-1)):end]);dims=2)[:]
storage_charge=sum(charge_df[!, colname] for colname in String.(storage_df.Resource))[start_time:end_time] ;
storage_discharge=sum(power[!, colname] for colname in String.(storage_df.Resource))[start_time:end_time] ;
thermal_power=sum(power[!, colname] for colname in String.(thermal_df.Resource))[start_time:end_time] ;
total_solar=sum(power[!, colname] for colname in String.(renewable_df.Resource) if renewable_dict[colname] == "solar")[start_time:end_time] ;
total_wind=sum(power[!, colname] for colname in String.(renewable_df.Resource) if renewable_dict[colname] == "wind")[start_time:end_time] ;
ren_curtail=sum(curtailment_df[!, colname] for colname in String.(renewable_df.Resource))[start_time:end_time] ;

x_data = 1:24
thermal_and_wind = thermal_power + total_wind
thermal_and_wind_and_solar = thermal_and_wind + total_solar
thermal_wind_solar_storage = thermal_and_wind_and_solar + storage_discharge
curtailment_data = thermal_wind_solar_storage + ren_curtail

plot(x_data, -storage_charge, seriestype=:path, fillrange=0, fillalpha=0.3, label="Storage In", xlabel="Time (hours)", ylabel="MegaWatts", color=:purple)
plot!(x_data, demand, seriestype=:path, label="Load", color=:black)
plot!(x_data, thermal_power, seriestype=:path, fillrange=0, fillalpha=0.3, label="CT", color=:black)
plot!(x_data, thermal_and_wind, seriestype=:path, fillrange=(thermal_data), fillalpha=0.2, label="Wind Data", color=:blue)
plot!(x_data, thermal_and_wind_and_solar, seriestype=:path, fillrange=(thermal_and_wind), fillalpha=0.3, label="Solar Data", color=:yellow)
plot!(x_data, thermal_wind_solar_storage, seriestype=:path, fillrange=(thermal_and_wind_and_solar), fillalpha=0.3, label="Storage out", color=:purple)
plot!(x_data, curtailment_data, seriestype=:path, fillrange=(thermal_wind_solar_storage), fillalpha=0.3, label="Curtailment", color=:red)


UndefVarError: UndefVarError: `thermal_data` not defined